# Q-table Debug Check (init/final + fix preview)

This notebook validates three things for one run:
1. Q-table structure should contain `2 * N` tables (`N` sellers x `{init, final}`).
2. Apply the same init-repair logic as `fix_initial_qtables.py` **in-memory**.
3. Show readable init/final Q-tables with `action_id -> action_name`.


In [4]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

import sys
project_root = Path('../..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import Config
from src.environment import PricingEnvironment
from src.agent import calculate_heuristic_init_values
from src.strategies import get_strategy_name

# ===== Parameters =====
RESULTS_ROOT = project_root / 'data' / 'results'   # change to data/results after full download
EXPERIMENT = 'scan_3strats_mu0.04_k40'
N_VALUE = 10
RUN_ID = 0

cfg_path = RESULTS_ROOT / EXPERIMENT / f'Config_N_{N_VALUE}.json'
run_path = RESULTS_ROOT / EXPERIMENT / f'N_{N_VALUE}' / f'run_{RUN_ID}.parquet'
qtable_path = RESULTS_ROOT / EXPERIMENT / f'N_{N_VALUE}' / f'run_{RUN_ID}_qtable.parquet'

print('cfg_path   =', cfg_path)
print('run_path   =', run_path)
print('qtable_path=', qtable_path)
print('run exists =', run_path.exists())
print('q exists   =', qtable_path.exists())

cfg_path   = /Users/liushijian/Documents/GitHub/Amazon_BuyBox_Reinforcement_Learning/pricing_marl/data/results/scan_3strats_mu0.04_k40/Config_N_10.json
run_path   = /Users/liushijian/Documents/GitHub/Amazon_BuyBox_Reinforcement_Learning/pricing_marl/data/results/scan_3strats_mu0.04_k40/N_10/run_0.parquet
qtable_path= /Users/liushijian/Documents/GitHub/Amazon_BuyBox_Reinforcement_Learning/pricing_marl/data/results/scan_3strats_mu0.04_k40/N_10/run_0_qtable.parquet
run exists = True
q exists   = True


In [5]:
import re

from src.strategies import (
    ACT_UNDERCUT,
    ACT_MATCH,
    ACT_ABOVE,
    ACT_UNDER_RESET,
)

SCAN_RE = re.compile(r'^scan_(3strats|4strats)_mu(.+)_k(.+)$')


def build_config_from_json(cfg_json: dict) -> Config:
    return Config(
        num_sellers=cfg_json['num_sellers'],
        num_grids=cfg_json['num_grids'],
        active_strategies=cfg_json['active_strategies'],
        a_val=cfg_json['a_val'],
        c_val=cfg_json['c_val'],
        a0=cfg_json['a0'],
        mu=cfg_json['mu'],
        xi=cfg_json['xi'],
        alpha=cfg_json['alpha'],
        gamma=cfg_json['gamma'],
        max_episodes=cfg_json['max_episodes'],
        beta=cfg_json['beta'],
        converge_period=cfg_json['converge_period'],
        K=cfg_json['K'],
        eval_H=cfg_json['eval_H'],
        save_training_data=cfg_json['save_training_data'],
    )


def build_config_from_experiment_name(experiment: str, n_value: int) -> Config:
    m = SCAN_RE.match(experiment)
    if not m:
        raise ValueError(f'Cannot parse experiment name: {experiment}')

    label = m.group(1)
    mu_val = float(m.group(2))
    k_val = int(float(m.group(3)))

    if label == '3strats':
        active_strategies = [ACT_UNDERCUT, ACT_MATCH, ACT_ABOVE]
    elif label == '4strats':
        active_strategies = [ACT_UNDERCUT, ACT_MATCH, ACT_ABOVE, ACT_UNDER_RESET]
    else:
        raise ValueError(f'Unsupported experiment label: {label}')

    # exp02 defaults used for compatibility when config JSON is unavailable
    return Config(
        num_sellers=n_value,
        num_grids=10,
        active_strategies=active_strategies,
        a_val=2.0,
        c_val=1.0,
        a0=0.0,
        mu=mu_val,
        xi=0.1,
        alpha=0.15,
        gamma=0.95,
        max_episodes=2_000_000,
        beta=1e-5,
        converge_period=100_000,
        K=k_val,
        eval_H=2_000,
        save_training_data=False,
    )


def resolve_config(cfg_path: Path, experiment: str, n_value: int) -> Config:
    if cfg_path.exists():
        with open(cfg_path, 'r') as f:
            cfg_json = json.load(f)
        print(f'[CFG] Using config file: {cfg_path.name}')
        return build_config_from_json(cfg_json)

    print(f'[WARN] Missing config file: {cfg_path.name}; fallback to experiment-name parsing')
    return build_config_from_experiment_name(experiment, n_value)


def build_expected_init_qtable(cfg: Config, run_id: int) -> pd.DataFrame:
    env = PricingEnvironment(cfg)
    init_q = calculate_heuristic_init_values(env, cfg)

    rows = []
    for seller_id in range(cfg.num_sellers):
        for state_idx in range(cfg.num_grids):
            for action_idx, action_id in enumerate(cfg.active_strategies):
                rows.append(
                    {
                        'run_id': int(run_id),
                        'seller_id': int(seller_id),
                        'state_idx': int(state_idx),
                        'action_idx': int(action_idx),
                        'action_id': int(action_id),
                        'q_value': float(init_q[state_idx, action_idx]),
                        'training_stop_episode': -1,
                        'qtable_type': 'init',
                    }
                )

    df = pd.DataFrame(rows)
    if not df.empty:
        df['run_id'] = df['run_id'].astype('int32')
        df['seller_id'] = df['seller_id'].astype('int16')
        df['state_idx'] = df['state_idx'].astype('int16')
        df['action_idx'] = df['action_idx'].astype('int16')
        df['action_id'] = df['action_id'].astype('int16')
        df['training_stop_episode'] = df['training_stop_episode'].astype('int32')
        df['q_value'] = df['q_value'].astype('float32')
    return df


def apply_init_fix_in_memory(qtable_df: pd.DataFrame, expected_init_df: pd.DataFrame) -> pd.DataFrame:
    if 'qtable_type' not in qtable_df.columns:
        raise ValueError('qtable_type column is required.')
    non_init = qtable_df[qtable_df['qtable_type'] != 'init'].copy()
    repaired = pd.concat([expected_init_df, non_init], ignore_index=True)
    for col in qtable_df.columns:
        if col not in repaired.columns:
            repaired[col] = pd.NA
    repaired = repaired[qtable_df.columns]
    return repaired

In [6]:
cfg = resolve_config(cfg_path, EXPERIMENT, N_VALUE)

if not run_path.exists():
    raise FileNotFoundError(f'Missing run parquet: {run_path}')
if not qtable_path.exists():
    raise FileNotFoundError(f'Missing qtable parquet: {qtable_path}')

q_raw = pd.read_parquet(qtable_path)
df_eval = pd.read_parquet(run_path)

print('Eval rows:', len(df_eval))
print('Q rows   :', len(q_raw))
print('Q columns:', q_raw.columns.tolist())
print('qtable_type counts:', q_raw['qtable_type'].value_counts().to_dict() if 'qtable_type' in q_raw.columns else 'NO qtable_type')

expected_num_tables = 2 * cfg.num_sellers
actual_num_tables = q_raw[['qtable_type', 'seller_id']].drop_duplicates().shape[0]
expected_rows_per_table = cfg.num_grids * cfg.num_actions

print('\n=== Structure check ===')
print('Expected #tables (2*N):', expected_num_tables)
print('Actual #tables        :', actual_num_tables)
print('Pass                  :', expected_num_tables == actual_num_tables)

sizes = q_raw.groupby(['qtable_type', 'seller_id']).size().unstack('qtable_type').fillna(0).astype(int)
print('\nExpected rows per (seller, table):', expected_rows_per_table)
print(sizes.to_string())

[CFG] Using config file: Config_N_10.json
Eval rows: 2000
Q rows   : 600
Q columns: ['run_id', 'seller_id', 'state_idx', 'action_idx', 'action_id', 'q_value', 'training_stop_episode', 'qtable_type']
qtable_type counts: {'init': 300, 'final': 300}

=== Structure check ===
Expected #tables (2*N): 20
Actual #tables        : 20
Pass                  : True

Expected rows per (seller, table): 30
qtable_type  final  init
seller_id               
0               30    30
1               30    30
2               30    30
3               30    30
4               30    30
5               30    30
6               30    30
7               30    30
8               30    30
9               30    30


In [7]:
expected_init = build_expected_init_qtable(cfg, RUN_ID)
q_fixed = apply_init_fix_in_memory(q_raw, expected_init).copy()
q_fixed['action_name'] = q_fixed['action_id'].map(get_strategy_name)

# Compare old vs fixed init
if 'qtable_type' in q_raw.columns:
    key = ['run_id', 'seller_id', 'state_idx', 'action_idx', 'action_id']
    old_init = q_raw[q_raw['qtable_type'] == 'init'][key + ['q_value']]
    new_init = q_fixed[q_fixed['qtable_type'] == 'init'][key + ['q_value']]
    m = old_init.merge(new_init, on=key, suffixes=('_old', '_new'))
    d = (m['q_value_old'] - m['q_value_new']).abs()
    print('Init replacement preview:')
    print('  max abs diff (old_init vs fixed_init):', float(d.max()))
    print('  changed cells:', int((d > 1e-9).sum()), '/', len(d))

# Compare fixed init vs final
key = ['run_id', 'seller_id', 'state_idx', 'action_idx', 'action_id']
m2 = (
    q_fixed[q_fixed['qtable_type'] == 'init'][key + ['q_value']]
    .merge(
        q_fixed[q_fixed['qtable_type'] == 'final'][key + ['q_value']],
        on=key,
        suffixes=('_init', '_final')
    )
)
d2 = (m2['q_value_final'] - m2['q_value_init']).abs()
print('\nFixed init vs final summary:')
print(d2.describe().to_string())

[Env] Loading cached table: lookup_N10_G10_mu0.04_a2.0_c1.0_a00.0_xi0.1_K40_strats0_1_2_stateLow_floorC.pkl
Init replacement preview:
  max abs diff (old_init vs fixed_init): 0.0
  changed cells: 0 / 300

Fixed init vs final summary:
count    300.000000
mean       0.093071
std        0.219149
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.229057


In [8]:
# Show per-seller init/final Q-table with action names
action_order = [get_strategy_name(aid) for aid in cfg.active_strategies]

for seller_id in range(cfg.num_sellers):
    print(f'\n===== Seller {seller_id} =====')
    init_tbl = (
        q_fixed[(q_fixed['seller_id'] == seller_id) & (q_fixed['qtable_type'] == 'init')]
        .pivot(index='state_idx', columns='action_name', values='q_value')
        .reindex(columns=action_order)
        .sort_index()
    )
    final_tbl = (
        q_fixed[(q_fixed['seller_id'] == seller_id) & (q_fixed['qtable_type'] == 'final')]
        .pivot(index='state_idx', columns='action_name', values='q_value')
        .reindex(columns=action_order)
        .sort_index()
    )

    print('INIT Q-table')
    display(init_tbl)
    print('FINAL Q-table')
    display(final_tbl)


===== Seller 0 =====
INIT Q-table


action_name,Undercut,Match,Above
state_idx,,,
0,0.000000,0.000112,0.041127
1,0.000002,0.006239,0.040006
2,0.006374,0.020878,0.039695
3,0.031588,0.037744,0.039480
4,0.075598,0.056837,0.039355
5,0.138406,0.078156,0.039319
6,0.220011,0.101693,0.039371
7,0.320393,0.127198,0.039496
8,0.438942,0.147617,0.039253


FINAL Q-table


action_name,Undercut,Match,Above
state_idx,,,
0,0.507359,0.507184,0.534954
1,0.000002,0.006239,0.040006
2,0.006374,0.020878,0.039695
3,0.031588,0.037744,0.039480
4,0.075598,0.056837,0.039355
5,0.138406,0.072375,0.039319
6,0.220011,0.101693,0.039371
7,0.320393,0.127198,0.039496
8,0.438942,0.147617,0.039253



===== Seller 1 =====
INIT Q-table


...

action_name,Undercut,Match,Above
state_idx,,,
0,0.507572,0.507330,0.534933
1,0.000002,0.006239,0.040006
2,0.006374,0.020878,0.039695
3,0.031588,0.037744,0.039480
4,0.075598,0.056837,0.039355
5,0.124684,0.078156,0.039319
6,0.220011,0.101693,0.039371
7,0.320393,0.127198,0.039496
8,0.438942,0.147617,0.039253



===== Seller 9 =====
INIT Q-table


action_name,Undercut,Match,Above
state_idx,,,
0,0.000000,0.000112,0.041127
1,0.000002,0.006239,0.040006
2,0.006374,0.020878,0.039695
3,0.031588,0.037744,0.039480
4,0.075598,0.056837,0.039355
5,0.138406,0.078156,0.039319
6,0.220011,0.101693,0.039371
7,0.320393,0.127198,0.039496
8,0.438942,0.147617,0.039253


FINAL Q-table


action_name,Undercut,Match,Above
state_idx,,,
0,0.507526,0.507951,0.534933
1,0.000002,0.006239,0.040006
2,0.006374,0.020878,0.039695
3,0.031588,0.037744,0.039480
4,0.075598,0.056837,0.039355
5,0.138406,0.078156,0.039576
6,0.220011,0.101693,0.039371
7,0.320393,0.127198,0.039496
8,0.438942,0.147617,0.039253
